# 02 — End-to-End HCP Data Pipeline

This notebook downloads selected HCP task-fMRI files, extracts motion-cleaned Schaefer ROI time series, converts event files into volume-aligned targets, and saves one processed run bundle per subject-task-run.

## Pipeline

```text
HCP S3
  -> select and download files
  -> validate NIfTI and motion regressors
  -> extract Schaefer-300 ROI time series
  -> parse HCP event files
  -> build fixed-lag and HRF targets
  -> save a processed run bundle
```

Each stage is implemented as a small function so that atlas, confound, target, and acquisition choices can be changed independently.

The primary initial modeling pair is:

```text
X = X_roi.npy
y = y_labels_hrf.npy
```


In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import boto3
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.glm.first_level import compute_regressor
from nilearn.maskers import NiftiLabelsMasker

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

print("boto3:", boto3.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("NiBabel:", nib.__version__)

boto3: 1.43.64
NumPy: 2.4.6
Pandas: 3.0.5
NiBabel: 5.4.2


## 1. Resolve project paths

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by searching upward."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "hcp_ya_s1200"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "hcp_ya_s1200" / "runs"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
CACHE_DIR = PROJECT_ROOT / ".cache" / "nilearn"

for path in (RAW_DATA_DIR, PROCESSED_DATA_DIR, MANIFEST_DIR, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed runs:", PROCESSED_DATA_DIR)

Project root: /Users/srinivasgovindasurampudi/Projects/neurolens-rag
Raw data: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200
Processed runs: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/processed/hcp_ya_s1200/runs


## 2. Configuration

Start with one subject, one task, and one run. Expand the lists only after the first processed bundle passes validation.

In [ ]:
AWS_PROFILE = "hcp"
AWS_REGION = "us-east-1"
BUCKET_NAME = "hcp-openaccess"
DATASET_PREFIX = "HCP_1200"
DATASET_RELEASE = "HCP-YA-S1200"

SUBJECT_IDS = ["100307"]
TASKS = ["MOTOR"]
RUNS = ["LR"]

DOWNLOAD_IF_MISSING = True
OVERWRITE_RAW_FILES = False

ATLAS_N_ROIS = 300
ATLAS_NETWORKS = 7
ATLAS_RESOLUTION_MM = 2

DETREND = True
STANDARDIZE = "zscore_sample"
STANDARDIZE_CONFOUNDS = True
LOW_PASS_HZ = None
HIGH_PASS_HZ = None

HEMODYNAMIC_LAG_SECONDS = 5.0
HRF_MODEL = "spm"
HRF_HARD_LABEL_THRESHOLD_FRACTION = 0.10

OVERWRITE_PROCESSED_RUN = False
RUN_PIPELINE = True

print("Subjects:", SUBJECT_IDS)
print("Tasks:", TASKS)
print("Runs:", RUNS)
print("Planned runs:", len(SUBJECT_IDS) * len(TASKS) * len(RUNS))

Subjects: ['100307']
Tasks: ['MOTOR']
Runs: ['LR']
Planned runs: 1


## 3. Explicit task-event mappings

In [ ]:
TASK_EVENT_FILE_MAPS: dict[str, dict[str, str]] = {
    "MOTOR": {
        "lh.txt": "left_hand",
        "rh.txt": "right_hand",
        "lf.txt": "left_foot",
        "rf.txt": "right_foot",
        "t.txt": "tongue",
    },
    "EMOTION": {
        "fear.txt": "faces",
        "neut.txt": "shapes",
    },
    "WM": {
        "0bk_body.txt": "0back_body",
        "0bk_faces.txt": "0back_faces",
        "0bk_places.txt": "0back_places",
        "0bk_tools.txt": "0back_tools",
        "2bk_body.txt": "2back_body",
        "2bk_faces.txt": "2back_faces",
        "2bk_places.txt": "2back_places",
        "2bk_tools.txt": "2back_tools",
    },
}

for task in TASKS:
    if task not in TASK_EVENT_FILE_MAPS:
        raise KeyError(f"No event mapping is defined for task {task!r}.")

TASK_EVENT_FILE_MAPS

{'MOTOR': {'lh.txt': 'left_hand',
  'rh.txt': 'right_hand',
  'lf.txt': 'left_foot',
  'rf.txt': 'right_foot',
  't.txt': 'tongue'},
 'EMOTION': {'fear.txt': 'faces', 'neut.txt': 'shapes'},
 'WM': {'0bk_body.txt': '0back_body',
  '0bk_faces.txt': '0back_faces',
  '0bk_places.txt': '0back_places',
  '0bk_tools.txt': '0back_tools',
  '2bk_body.txt': '2back_body',
  '2bk_faces.txt': '2back_faces',
  '2bk_places.txt': '2back_places',
  '2bk_tools.txt': '2back_tools'}}

## 4. Run specifications

In [ ]:
@dataclass(frozen=True)
class RunSpec:
    subject_id: str
    task: str
    run: str

    @property
    def run_name(self) -> str:
        return f"tfMRI_{self.task}_{self.run}"

    @property
    def s3_prefix(self) -> str:
        return (
            f"{DATASET_PREFIX}/{self.subject_id}/"
            f"MNINonLinear/Results/{self.run_name}/"
        )

    @property
    def raw_run_dir(self) -> Path:
        return (
            RAW_DATA_DIR / self.subject_id / "MNINonLinear" / "Results" / self.run_name
        )

    @property
    def func_path(self) -> Path:
        return self.raw_run_dir / f"{self.run_name}.nii.gz"

    @property
    def movement_path(self) -> Path:
        return self.raw_run_dir / "Movement_Regressors.txt"

    @property
    def events_dir(self) -> Path:
        return self.raw_run_dir / "EVs"

    @property
    def output_dir(self) -> Path:
        return PROCESSED_DATA_DIR / f"sub-{self.subject_id}" / self.run_name


@dataclass
class RunResult:
    subject_id: str
    task: str
    run: str
    status: str
    output_dir: str | None
    n_timepoints: int | None
    n_rois: int | None
    n_conditions: int | None
    message: str


def build_run_specs(
    subject_ids: Iterable[str],
    tasks: Iterable[str],
    runs: Iterable[str],
) -> list[RunSpec]:
    return [
        RunSpec(str(subject_id), str(task), str(run))
        for subject_id in subject_ids
        for task in tasks
        for run in runs
    ]


RUN_SPECS = build_run_specs(SUBJECT_IDS, TASKS, RUNS)
pd.DataFrame([asdict(spec) for spec in RUN_SPECS])

,subject_id,task,run
0,100307,MOTOR,LR


## 5. AWS connection

In [ ]:
def create_hcp_s3_client(*, profile_name: str, region_name: str):
    """Create and validate an S3 client using a named profile."""
    try:
        session = boto3.Session(profile_name=profile_name, region_name=region_name)
    except ProfileNotFound as error:
        raise RuntimeError(f"AWS profile {profile_name!r} was not found.") from error

    credentials = session.get_credentials()
    if credentials is None:
        raise RuntimeError(f"No credentials resolved for profile {profile_name!r}.")

    client = session.client("s3", region_name=region_name)

    try:
        response = client.list_objects_v2(
            Bucket=BUCKET_NAME,
            Prefix=f"{DATASET_PREFIX}/",
            Delimiter="/",
            MaxKeys=5,
        )
    except NoCredentialsError as error:
        raise RuntimeError("AWS credentials were not found.") from error
    except ClientError as error:
        details = error.response.get("Error", {})
        raise RuntimeError(
            f"AWS request failed: {details.get('Code', 'Unknown')}: "
            f"{details.get('Message', 'No AWS message.')}"
        ) from error

    print("Connected to:", BUCKET_NAME)
    print("Credential source:", credentials.method)
    print("Example prefixes:", [x["Prefix"] for x in response.get("CommonPrefixes", [])])
    return client


s3 = create_hcp_s3_client(profile_name=AWS_PROFILE, region_name=AWS_REGION)

Connected to: hcp-openaccess
Credential source: shared-credentials-file
Example prefixes: ['HCP_1200/100206/', 'HCP_1200/100307/', 'HCP_1200/100408/', 'HCP_1200/100610/']


## 6. Acquisition functions

In [ ]:
def list_s3_objects(*, client, bucket: str, prefix: str) -> pd.DataFrame:
    """List all objects under an S3 prefix."""
    paginator = client.get_paginator("list_objects_v2")
    rows: list[dict[str, Any]] = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            rows.append(
                {
                    "key": item["Key"],
                    "size_bytes": int(item["Size"]),
                    "size_mb": float(item["Size"]) / (1024**2),
                    "last_modified": item["LastModified"],
                }
            )

    return pd.DataFrame(rows)


def select_required_run_objects(objects: pd.DataFrame, *, spec: RunSpec) -> pd.DataFrame:
    """Select functional NIfTI, movement regressors, and EV files."""
    if objects.empty:
        return objects.copy()

    suffixes = (
        f"/{spec.run_name}.nii.gz",
        "/Movement_Regressors.txt",
        "/Movement_Regressors_dt.txt",
    )

    mask = objects["key"].str.endswith(suffixes) | objects["key"].str.contains("/EVs/")
    return objects.loc[mask].sort_values("key").reset_index(drop=True)


def local_path_for_s3_key(key: str) -> Path:
    relative_path = Path(key).relative_to(DATASET_PREFIX)
    return RAW_DATA_DIR / relative_path


def inspect_run_download(*, client, spec: RunSpec) -> pd.DataFrame:
    """Build the remote acquisition table for one run."""
    objects = list_s3_objects(client=client, bucket=BUCKET_NAME, prefix=spec.s3_prefix)
    selected = select_required_run_objects(objects, spec=spec)

    if selected.empty:
        raise FileNotFoundError(f"No S3 objects found for {spec.s3_prefix}")

    selected = selected.copy()
    selected["local_path"] = selected["key"].map(lambda key: str(local_path_for_s3_key(key)))
    return selected


def download_run_files(*, client, selected_objects: pd.DataFrame, overwrite: bool) -> pd.DataFrame:
    """Download required objects and verify local byte sizes."""
    rows: list[dict[str, Any]] = []

    for row in selected_objects.itertuples(index=False):
        local_path = Path(row.local_path)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        remote_size = int(row.size_bytes)
        local_size_before = local_path.stat().st_size if local_path.exists() else None

        if local_path.exists() and not overwrite and local_size_before == remote_size:
            status = "skipped_valid_existing"
        else:
            print("Downloading:", row.key)
            client.download_file(BUCKET_NAME, row.key, str(local_path))
            status = "downloaded"

        local_size_after = local_path.stat().st_size if local_path.exists() else None
        rows.append(
            {
                "key": row.key,
                "local_path": str(local_path),
                "remote_size_bytes": remote_size,
                "local_size_bytes": local_size_after,
                "status": status,
                "size_matches": local_size_after == remote_size,
            }
        )

    result = pd.DataFrame(rows)
    if not result["size_matches"].all():
        raise RuntimeError("At least one downloaded file failed size verification.")
    return result

## 7. NIfTI and motion-regressor functions

In [ ]:
def inspect_nifti(func_path: Path) -> tuple[nib.Nifti1Image, dict[str, Any]]:
    """Load a NIfTI lazily and validate basic metadata."""
    img = nib.load(func_path)

    if len(img.shape) != 4:
        raise ValueError(f"Expected 4D fMRI, got shape {img.shape}.")

    zooms = img.header.get_zooms()
    metadata = {
        "shape": [int(value) for value in img.shape],
        "n_timepoints": int(img.shape[-1]),
        "voxel_sizes_mm": [float(value) for value in zooms[:3]],
        "tr_seconds": float(zooms[3]),
        "dtype": str(img.get_data_dtype()),
        "affine_is_finite": bool(np.isfinite(img.affine).all()),
    }

    if not metadata["affine_is_finite"]:
        raise ValueError("NIfTI affine contains non-finite values.")

    return img, metadata


def load_motion_regressors(movement_path: Path, *, expected_timepoints: int) -> np.ndarray:
    """Load and validate HCP movement regressors."""
    movement = np.loadtxt(movement_path)
    if movement.ndim == 1:
        movement = movement[:, None]

    if movement.shape[0] != expected_timepoints:
        raise ValueError(
            f"Motion regressors have {movement.shape[0]} rows; "
            f"expected {expected_timepoints}."
        )

    if not np.isfinite(movement).all():
        raise ValueError("Motion regressors contain non-finite values.")

    return movement

## 8. Atlas and ROI extraction functions

In [ ]:
def fetch_schaefer_atlas(*, n_rois: int, yeo_networks: int, resolution_mm: int):
    """Fetch and validate the configured Schaefer atlas."""
    atlas = fetch_atlas_schaefer_2018(
        n_rois=n_rois,
        yeo_networks=yeo_networks,
        resolution_mm=resolution_mm,
    )

    labels = [label.decode("utf-8") if isinstance(label, bytes) else str(label) for label in atlas.labels]
    if len(labels) == n_rois + 1:
        labels = labels[1:]
    if len(labels) != n_rois:
        raise ValueError(f"Atlas label count {len(labels)} does not match {n_rois}.")

    return Path(atlas.maps), labels


def build_roi_masker(*, atlas_path: Path, roi_labels: list[str], tr_seconds: float) -> NiftiLabelsMasker:
    """Build the configured motion-cleaned ROI extraction pipeline."""
    return NiftiLabelsMasker(
        labels_img=str(atlas_path),
        labels=roi_labels,
        background_label=0,
        standardize=STANDARDIZE,
        standardize_confounds=STANDARDIZE_CONFOUNDS,
        detrend=DETREND,
        low_pass=LOW_PASS_HZ,
        high_pass=HIGH_PASS_HZ,
        t_r=tr_seconds,
        resampling_target="data",
        memory=str(CACHE_DIR),
        memory_level=1,
        verbose=1,
    )


def extract_motion_cleaned_roi_timeseries(
    *,
    func_path: Path,
    movement: np.ndarray,
    atlas_path: Path,
    roi_labels: list[str],
    tr_seconds: float,
) -> np.ndarray:
    """Extract a time-by-ROI matrix after motion regression."""
    masker = build_roi_masker(
        atlas_path=atlas_path,
        roi_labels=roi_labels,
        tr_seconds=tr_seconds,
    )

    roi_timeseries = np.asarray(
        masker.fit_transform(str(func_path), confounds=movement),
        dtype=np.float32,
    )

    if roi_timeseries.ndim != 2:
        raise ValueError(f"ROI matrix must be 2D, got {roi_timeseries.shape}.")
    if not np.isfinite(roi_timeseries).all():
        raise ValueError("ROI matrix contains non-finite values.")

    zero_variance_rois = np.where(roi_timeseries.std(axis=0) < 1e-8)[0]
    if len(zero_variance_rois) > 0:
        raise ValueError(f"Zero-variance ROIs: {zero_variance_rois.tolist()}")

    return roi_timeseries

## 9. Event parsing functions

In [ ]:
def read_hcp_event_file(path: Path, *, condition: str) -> pd.DataFrame:
    """Read one three-column HCP EV file."""
    events = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        names=["onset", "duration", "amplitude"],
        usecols=[0, 1, 2],
        dtype=float,
    )

    if events.empty:
        raise ValueError(f"Event file is empty: {path}")

    values = events[["onset", "duration", "amplitude"]].to_numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Non-finite event values in {path}")
    if (events["onset"] < 0).any():
        raise ValueError(f"Negative event onset in {path}")
    if (events["duration"] <= 0).any():
        raise ValueError(f"Non-positive event duration in {path}")

    events["condition"] = condition
    events["source_file"] = path.name
    events["offset"] = events["onset"] + events["duration"]
    return events


def load_task_events(
    *,
    events_dir: Path,
    event_file_map: dict[str, str],
    run_duration_seconds: float,
    tr_seconds: float,
) -> pd.DataFrame:
    """Load and validate all modeled event files for a task."""
    frames: list[pd.DataFrame] = []

    for filename, condition in event_file_map.items():
        path = events_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Expected event file not found: {path}")
        frames.append(read_hcp_event_file(path, condition=condition))

    events = pd.concat(frames, ignore_index=True).sort_values(
        ["onset", "condition"]
    ).reset_index(drop=True)

    if events["offset"].max() > run_duration_seconds + tr_seconds:
        raise ValueError("At least one event extends beyond the run.")

    return events

## 10. Target construction functions

In [ ]:
def events_to_boxcar_matrix(
    events: pd.DataFrame,
    *,
    frame_times: np.ndarray,
    condition_to_column: dict[str, int],
) -> np.ndarray:
    matrix = np.zeros((len(frame_times), len(condition_to_column)), dtype=np.float32)

    for event in events.itertuples(index=False):
        column = condition_to_column[event.condition]
        active = (frame_times >= float(event.onset)) & (frame_times < float(event.offset))
        matrix[active, column] = float(event.amplitude)

    return matrix


def shift_target_matrix(matrix: np.ndarray, *, lag_volumes: int) -> tuple[np.ndarray, np.ndarray]:
    if lag_volumes < 0:
        raise ValueError("lag_volumes must be non-negative.")

    shifted = np.zeros_like(matrix)
    valid_mask = np.ones(matrix.shape[0], dtype=bool)

    if lag_volumes == 0:
        shifted[:] = matrix
        return shifted, valid_mask
    if lag_volumes >= matrix.shape[0]:
        raise ValueError("Hemodynamic lag exceeds run length.")

    shifted[lag_volumes:] = matrix[:-lag_volumes]
    valid_mask[:lag_volumes] = False
    return shifted, valid_mask


def events_to_hrf_matrix(
    events: pd.DataFrame,
    *,
    frame_times: np.ndarray,
    condition_names: list[str],
    hrf_model: str,
) -> np.ndarray:
    columns: list[np.ndarray] = []

    for condition in condition_names:
        condition_events = events[events["condition"] == condition]
        experimental_condition = np.vstack(
            [
                condition_events["onset"].to_numpy(dtype=float),
                condition_events["duration"].to_numpy(dtype=float),
                condition_events["amplitude"].to_numpy(dtype=float),
            ]
        )

        regressor, names = compute_regressor(
            experimental_condition,
            hrf_model=hrf_model,
            frame_times=frame_times,
            con_id=condition,
        )

        if regressor.shape[1] != 1:
            raise ValueError(f"Expected one HRF regressor for {condition}, got {names}.")
        columns.append(regressor[:, 0])

    matrix = np.column_stack(columns).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("HRF matrix contains non-finite values.")
    return matrix


def condition_matrix_to_labels(matrix: np.ndarray, *, threshold: float) -> tuple[np.ndarray, np.ndarray]:
    winning_columns = np.argmax(matrix, axis=1)
    winning_values = np.max(matrix, axis=1)
    labels = winning_columns.astype(np.int64) + 1
    task_mask = winning_values > threshold
    labels[~task_mask] = 0
    return labels, task_mask


def build_targets(
    *,
    events: pd.DataFrame,
    n_timepoints: int,
    tr_seconds: float,
    condition_names: list[str],
) -> dict[str, Any]:
    frame_times = np.arange(n_timepoints, dtype=np.float64) * tr_seconds
    condition_to_column = {condition: index for index, condition in enumerate(condition_names)}
    condition_to_class = {condition: index + 1 for index, condition in enumerate(condition_names)}
    class_to_condition = {0: "baseline", **{v: k for k, v in condition_to_class.items()}}

    boxcar = events_to_boxcar_matrix(
        events,
        frame_times=frame_times,
        condition_to_column=condition_to_column,
    )

    lag_volumes = int(round(HEMODYNAMIC_LAG_SECONDS / tr_seconds))
    fixed_lag, valid_mask_fixed_lag = shift_target_matrix(
        boxcar,
        lag_volumes=lag_volumes,
    )

    hrf = events_to_hrf_matrix(
        events,
        frame_times=frame_times,
        condition_names=condition_names,
        hrf_model=HRF_MODEL,
    )

    labels_fixed_lag, task_mask_fixed_lag = condition_matrix_to_labels(
        fixed_lag,
        threshold=0.0,
    )

    hrf_threshold = HRF_HARD_LABEL_THRESHOLD_FRACTION * float(hrf.max())
    labels_hrf, task_mask_hrf = condition_matrix_to_labels(
        hrf,
        threshold=hrf_threshold,
    )

    return {
        "frame_times": frame_times.astype(np.float32),
        "boxcar": boxcar.astype(np.float32),
        "fixed_lag": fixed_lag.astype(np.float32),
        "hrf": hrf.astype(np.float32),
        "labels_fixed_lag": labels_fixed_lag,
        "labels_hrf": labels_hrf,
        "valid_mask_fixed_lag": valid_mask_fixed_lag,
        "valid_mask_hrf": np.ones(n_timepoints, dtype=bool),
        "task_mask_fixed_lag": task_mask_fixed_lag,
        "task_mask_hrf": task_mask_hrf,
        "condition_to_column": condition_to_column,
        "condition_to_class": condition_to_class,
        "class_to_condition": class_to_condition,
        "lag_volumes": lag_volumes,
        "effective_lag_seconds": lag_volumes * tr_seconds,
        "hrf_threshold": hrf_threshold,
    }

## 11. Bundle validation and saving

In [ ]:
def validate_processed_bundle(
    *,
    roi_timeseries: np.ndarray,
    targets: dict[str, Any],
    n_rois: int,
) -> dict[str, Any]:
    n_timepoints = roi_timeseries.shape[0]

    if roi_timeseries.shape[1] != n_rois:
        raise ValueError(f"Expected {n_rois} ROIs, got {roi_timeseries.shape[1]}.")

    for name in ("boxcar", "fixed_lag", "hrf"):
        array = targets[name]
        if array.shape[0] != n_timepoints:
            raise ValueError(f"{name} time dimension does not match X.")
        if not np.isfinite(array).all():
            raise ValueError(f"{name} contains non-finite values.")

    for name in (
        "labels_fixed_lag",
        "labels_hrf",
        "valid_mask_fixed_lag",
        "valid_mask_hrf",
        "task_mask_fixed_lag",
        "task_mask_hrf",
        "frame_times",
    ):
        if targets[name].shape != (n_timepoints,):
            raise ValueError(f"{name} has incompatible shape {targets[name].shape}.")

    return {
        "n_timepoints": int(n_timepoints),
        "n_rois": int(roi_timeseries.shape[1]),
        "n_conditions": int(targets["hrf"].shape[1]),
        "roi_mean_absolute_mean": float(np.abs(roi_timeseries.mean(axis=0)).mean()),
        "roi_mean_std": float(roi_timeseries.std(axis=0).mean()),
        "fixed_lag_task_volumes": int(targets["task_mask_fixed_lag"].sum()),
        "hrf_task_volumes": int(targets["task_mask_hrf"].sum()),
    }


def save_processed_bundle(
    *,
    spec: RunSpec,
    roi_timeseries: np.ndarray,
    roi_labels: list[str],
    movement: np.ndarray,
    events: pd.DataFrame,
    targets: dict[str, Any],
    nifti_metadata: dict[str, Any],
    validation_summary: dict[str, Any],
    acquisition_records: pd.DataFrame,
) -> Path:
    output_dir = spec.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    arrays = {
        "X_roi.npy": roi_timeseries.astype(np.float32),
        "movement_regressors.npy": movement.astype(np.float32),
        "frame_times.npy": targets["frame_times"],
        "y_boxcar.npy": targets["boxcar"],
        "y_fixed_lag.npy": targets["fixed_lag"],
        "y_hrf.npy": targets["hrf"],
        "y_labels_fixed_lag.npy": targets["labels_fixed_lag"],
        "y_labels_hrf.npy": targets["labels_hrf"],
        "valid_mask_fixed_lag.npy": targets["valid_mask_fixed_lag"],
        "valid_mask_hrf.npy": targets["valid_mask_hrf"],
        "task_mask_fixed_lag.npy": targets["task_mask_fixed_lag"],
        "task_mask_hrf.npy": targets["task_mask_hrf"],
    }

    for filename, array in arrays.items():
        np.save(output_dir / filename, array)

    pd.DataFrame(
        {"roi_index": np.arange(len(roi_labels)), "roi_label": roi_labels}
    ).to_csv(output_dir / "roi_labels.tsv", sep="	", index=False)

    events.to_csv(output_dir / "events_long.tsv", sep="	", index=False)
    acquisition_records.to_csv(
        output_dir / "acquisition_records.tsv", sep="	", index=False
    )

    volume_targets = pd.DataFrame(
        {
            "volume_index": np.arange(roi_timeseries.shape[0]),
            "frame_time_seconds": targets["frame_times"],
            "label_fixed_lag": targets["labels_fixed_lag"],
            "label_hrf": targets["labels_hrf"],
            "valid_fixed_lag": targets["valid_mask_fixed_lag"],
            "valid_hrf": targets["valid_mask_hrf"],
            "task_fixed_lag": targets["task_mask_fixed_lag"],
            "task_hrf": targets["task_mask_hrf"],
        }
    )

    condition_names = list(targets["condition_to_column"].keys())
    for index, condition in enumerate(condition_names):
        volume_targets[f"boxcar_{condition}"] = targets["boxcar"][:, index]
        volume_targets[f"fixed_lag_{condition}"] = targets["fixed_lag"][:, index]
        volume_targets[f"hrf_{condition}"] = targets["hrf"][:, index]

    volume_targets.to_csv(output_dir / "volume_targets.tsv", sep="	", index=False)

    metadata = {
        "dataset_release": DATASET_RELEASE,
        "subject_id": spec.subject_id,
        "task": spec.task,
        "run": spec.run,
        "run_name": spec.run_name,
        "source_nifti": str(spec.func_path.relative_to(PROJECT_ROOT)),
        "movement_regressors": str(spec.movement_path.relative_to(PROJECT_ROOT)),
        "events_directory": str(spec.events_dir.relative_to(PROJECT_ROOT)),
        "atlas": {
            "name": "Schaefer2018",
            "n_rois": ATLAS_N_ROIS,
            "yeo_networks": ATLAS_NETWORKS,
            "resolution_mm": ATLAS_RESOLUTION_MM,
        },
        "roi_preprocessing": {
            "detrend": DETREND,
            "standardize": STANDARDIZE,
            "standardize_confounds": STANDARDIZE_CONFOUNDS,
            "confounds": "Movement_Regressors.txt",
            "low_pass_hz": LOW_PASS_HZ,
            "high_pass_hz": HIGH_PASS_HZ,
        },
        "target_construction": {
            "hrf_model": HRF_MODEL,
            "requested_lag_seconds": HEMODYNAMIC_LAG_SECONDS,
            "lag_volumes": targets["lag_volumes"],
            "effective_lag_seconds": targets["effective_lag_seconds"],
            "hrf_threshold_fraction": HRF_HARD_LABEL_THRESHOLD_FRACTION,
            "hrf_threshold_value": targets["hrf_threshold"],
            "condition_to_column": targets["condition_to_column"],
            "condition_to_class": targets["condition_to_class"],
            "class_to_condition": {str(k): v for k, v in targets["class_to_condition"].items()},
        },
        "nifti": nifti_metadata,
        "validation": validation_summary,
        "primary_modeling_pair": {
            "X": "X_roi.npy",
            "y": "y_labels_hrf.npy",
            "valid_mask": "valid_mask_hrf.npy",
        },
    }

    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    return output_dir

## 12. End-to-end run processor

In [ ]:
def process_one_run(
    *,
    spec: RunSpec,
    client,
    atlas_path: Path,
    roi_labels: list[str],
    download_if_missing: bool,
    overwrite_raw_files: bool,
    overwrite_processed_run: bool,
) -> RunResult:
    """Run acquisition, ROI extraction, target generation, and saving."""
    print("" + "=" * 80)
    print(f"Processing subject={spec.subject_id}, task={spec.task}, run={spec.run}")
    print("=" * 80)

    metadata_path = spec.output_dir / "metadata.json"
    if metadata_path.exists() and not overwrite_processed_run:
        return RunResult(
            spec.subject_id,
            spec.task,
            spec.run,
            "skipped_existing_processed",
            str(spec.output_dir),
            None,
            None,
            None,
            "Processed metadata already exists.",
        )

    try:
        acquisition_records = inspect_run_download(client=client, spec=spec)

        raw_ready = (
            spec.func_path.exists()
            and spec.movement_path.exists()
            and spec.events_dir.exists()
        )

        if not raw_ready and not download_if_missing:
            raise FileNotFoundError(
                "Required raw files are missing and DOWNLOAD_IF_MISSING is False."
            )

        if download_if_missing:
            acquisition_records = download_run_files(
                client=client,
                selected_objects=acquisition_records,
                overwrite=overwrite_raw_files,
            )

        _, nifti_metadata = inspect_nifti(spec.func_path)
        n_timepoints = nifti_metadata["n_timepoints"]
        tr_seconds = nifti_metadata["tr_seconds"]

        movement = load_motion_regressors(
            spec.movement_path,
            expected_timepoints=n_timepoints,
        )

        roi_timeseries = extract_motion_cleaned_roi_timeseries(
            func_path=spec.func_path,
            movement=movement,
            atlas_path=atlas_path,
            roi_labels=roi_labels,
            tr_seconds=tr_seconds,
        )

        events = load_task_events(
            events_dir=spec.events_dir,
            event_file_map=TASK_EVENT_FILE_MAPS[spec.task],
            run_duration_seconds=n_timepoints * tr_seconds,
            tr_seconds=tr_seconds,
        )

        targets = build_targets(
            events=events,
            n_timepoints=n_timepoints,
            tr_seconds=tr_seconds,
            condition_names=list(TASK_EVENT_FILE_MAPS[spec.task].values()),
        )

        validation_summary = validate_processed_bundle(
            roi_timeseries=roi_timeseries,
            targets=targets,
            n_rois=ATLAS_N_ROIS,
        )

        output_dir = save_processed_bundle(
            spec=spec,
            roi_timeseries=roi_timeseries,
            roi_labels=roi_labels,
            movement=movement,
            events=events,
            targets=targets,
            nifti_metadata=nifti_metadata,
            validation_summary=validation_summary,
            acquisition_records=acquisition_records,
        )

        return RunResult(
            spec.subject_id,
            spec.task,
            spec.run,
            "completed",
            str(output_dir),
            validation_summary["n_timepoints"],
            validation_summary["n_rois"],
            validation_summary["n_conditions"],
            "Processed successfully.",
        )

    except Exception as error:
        return RunResult(
            spec.subject_id,
            spec.task,
            spec.run,
            "failed",
            None,
            None,
            None,
            None,
            f"{type(error).__name__}: {error}",
        )

## 13. Fetch the atlas once

In [ ]:
atlas_path, roi_labels = fetch_schaefer_atlas(
    n_rois=ATLAS_N_ROIS,
    yeo_networks=ATLAS_NETWORKS,
    resolution_mm=ATLAS_RESOLUTION_MM,
)

print("Atlas:", atlas_path)
print("ROI labels:", len(roi_labels))
print("First labels:", roi_labels[:5])

[fetch_atlas_schaefer_2018] Dataset directory found: /Users/srinivasgovindasurampudi/nilearn_data/schaefer_2018

Atlas: /Users/srinivasgovindasurampudi/nilearn_data/schaefer_2018/Schaefer2018_300Parcels_7Networks_order_FSLMNI152_2mm.nii.gz
ROI labels: 300
First labels: ['7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4', '7Networks_LH_Vis_5']


## 14. Preview acquisition size

In [ ]:
preview_frames: list[pd.DataFrame] = []

for spec in RUN_SPECS:
    preview = inspect_run_download(client=s3, spec=spec).copy()
    preview["subject_id"] = spec.subject_id
    preview["task"] = spec.task
    preview["run"] = spec.run
    preview_frames.append(preview)

acquisition_preview_df = pd.concat(preview_frames, ignore_index=True)

preview_summary_df = (
    acquisition_preview_df
    .groupby(["subject_id", "task", "run"], as_index=False)
    .agg(
        files=("key", "count"),
        size_gb=("size_bytes", lambda values: values.sum() / (1024**3)),
    )
)

preview_summary_df

,subject_id,task,run,files,size_gb
0,100307,MOTOR,LR,10,0.219146


In [ ]:
total_preview_gb = acquisition_preview_df["size_bytes"].sum() / (1024**3)
print(f"Total selected remote size: {total_preview_gb:.2f} GB")

display(
    acquisition_preview_df[
        ["subject_id", "task", "run", "size_mb", "key", "local_path"]
    ].sort_values(["subject_id", "task", "run", "key"])
)

Total selected remote size: 0.22 GB


,subject_id,task,run,size_mb,key,local_path
0,100307,MOTOR,LR,0.000006,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/Sync.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/Sync.txt
1,100307,MOTOR,LR,0.000106,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/cue.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/cue.txt
2,100307,MOTOR,LR,0.000024,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/lf.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/lf.txt
3,100307,MOTOR,LR,0.000024,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/lh.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/lh.txt
4,100307,MOTOR,LR,0.000024,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/rf.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/rf.txt
5,100307,MOTOR,LR,0.000024,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/rh.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/rh.txt
6,100307,MOTOR,LR,0.000024,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/t.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/EVs/t.txt
7,100307,MOTOR,LR,0.036022,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/Movement_Regressors.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/Movement_Regressors.txt
8,100307,MOTOR,LR,0.036022,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/Movement_Regressors_dt.txt,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/Movement_Regressors_dt.txt
9,100307,MOTOR,LR,224.333204,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/tfMRI_MOTOR_LR.nii.gz,/Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/raw/hcp_ya_s1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/tfMRI_MOTOR_LR.nii.gz


## 15. Run the pipeline

Keep `RUN_PIPELINE = False` until the acquisition preview is correct.

In [ ]:
if RUN_PIPELINE:
    run_results: list[RunResult] = []

    for spec in RUN_SPECS:
        result = process_one_run(
            spec=spec,
            client=s3,
            atlas_path=atlas_path,
            roi_labels=roi_labels,
            download_if_missing=DOWNLOAD_IF_MISSING,
            overwrite_raw_files=OVERWRITE_RAW_FILES,
            overwrite_processed_run=OVERWRITE_PROCESSED_RUN,
        )
        run_results.append(result)
        print(result.status, result.message)

    run_results_df = pd.DataFrame([asdict(result) for result in run_results])
    display(run_results_df)
else:
    run_results_df = pd.DataFrame()
    print("Pipeline not executed.")
    print("Review the preview, then set RUN_PIPELINE = True.")

Pipeline not executed.
Review the preview, then set RUN_PIPELINE = True.


## 16. Save the processed-run manifest

In [ ]:
if not run_results_df.empty:
    run_manifest_path = MANIFEST_DIR / "hcp_s1200_processed_runs.csv"
    run_results_df.to_csv(run_manifest_path, index=False)
    print("Saved:", run_manifest_path)

    failed_runs = run_results_df[run_results_df["status"] == "failed"]
    if not failed_runs.empty:
        print("Failed runs:")
        display(failed_runs)
else:
    print("No run manifest saved because the pipeline has not executed.")

No run manifest saved because the pipeline has not executed.


## 17. Inspect one completed bundle

In [ ]:
completed_specs = [
    spec
    for spec in RUN_SPECS
    if (spec.output_dir / "metadata.json").exists()
]

if completed_specs:
    example_spec = completed_specs[0]

    X = np.load(example_spec.output_dir / "X_roi.npy", mmap_mode="r")
    y = np.load(example_spec.output_dir / "y_labels_hrf.npy", mmap_mode="r")
    y_hrf = np.load(example_spec.output_dir / "y_hrf.npy", mmap_mode="r")
    metadata = json.loads(
        (example_spec.output_dir / "metadata.json").read_text(encoding="utf-8")
    )

    print("Example run:", example_spec)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("y_hrf shape:", y_hrf.shape)
    print("Primary pair:", metadata["primary_modeling_pair"])
else:
    print("No completed processed bundle found.")

No completed processed bundle found.


## 18. Quick QC

In [ ]:
if completed_specs:
    plt.figure(figsize=(14, 5))
    plt.plot(np.asarray(X[:, : min(8, X.shape[1])]))
    plt.xlabel("Volume")
    plt.ylabel("Standardized ROI signal")
    plt.title("Motion-cleaned ROI time series")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 3))
    plt.step(np.arange(len(y)), np.asarray(y), where="post")
    plt.xlabel("Volume")
    plt.ylabel("Class ID")
    plt.title("HRF-derived hard labels")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 5))
    plt.imshow(np.asarray(y_hrf).T, aspect="auto", origin="lower")
    plt.xlabel("Volume")
    plt.ylabel("Condition")
    plt.title("Continuous HRF target matrix")
    plt.colorbar(label="HRF amplitude")
    plt.tight_layout()
    plt.show()

## 19. Component boundaries for future experiments

### Acquisition

```python
list_s3_objects()
select_required_run_objects()
download_run_files()
```

### ROI preprocessing

```python
load_motion_regressors()
build_roi_masker()
extract_motion_cleaned_roi_timeseries()
```

### Event and target processing

```python
read_hcp_event_file()
load_task_events()
events_to_boxcar_matrix()
events_to_hrf_matrix()
build_targets()
```

### Orchestration and saving

```python
process_one_run()
save_processed_bundle()
```

These functions are deliberately notebook-local for now. After the pipeline is validated, they can move into modules such as:

```text
src/neurolens/data/acquisition.py
src/neurolens/data/roi.py
src/neurolens/data/events.py
src/neurolens/data/pipeline.py
```

The next notebook will be:

```text
03_dataset_dataloaders.ipynb
```

It will decide the temporal sample definition, window length, stride, label strategy, masks, subject-level splitting, and PyTorch batch shapes.
